# P03 — Exploração de EEG/ERP: Primeiros Passos

**Objetivo:** Este notebook introduz a exploração de dados de EEG usando MNE-Python. Vamos:

1. Carregar dados brutos
2. Visualizar sinais e eventos
3. Aplicar pré-processamento
4. Plotar ERPs (grand average)
5. Comparar condições

**Pré-requisitos:** `pip install mne numpy pandas matplotlib scipy`

In [ ]:
# Setup
import sys
from pathlib import Path

# Adicionar path do projeto
sys.path.insert(0, str(Path('..').resolve()))
from setup import PATHS, LOG, EEG_CONFIG, ERP_COMPONENTS

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Para salvar PNGs
import matplotlib.pyplot as plt

import mne
mne.set_log_level('WARNING')  # Reduzir verbosidade

print(f"MNE versão: {mne.__version__}")
print(f"Configuração EEG: {EEG_CONFIG}")

In [ ]:
# 1. Criar dados sintéticos para demonstração
print("\n=== Criando dados sintéticos ===")
np.random.seed(42)

sfreq = 250  # Hz
n_channels = 32
duration = 120  # segundos (2 min)
n_samples = int(sfreq * duration)

# Nomes dos canais no sistema 10-20
ch_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4',
            'O1', 'O2', 'F7', 'F8', 'T7', 'T8', 'P7', 'P8',
            'Fz', 'Cz', 'Pz', 'Oz', 'FC1', 'FC2', 'CP1', 'CP2',
            'AF3', 'AF4', 'PO3', 'PO4', 'F5', 'F6', 'C5', 'C6']
ch_types = ['eeg'] * n_channels

# Dados com ruído
data = np.random.randn(n_channels, n_samples) * 1e-6  # µV

# Adicionar um pouco de atividade alfa (8-12 Hz) em Oz, O1, O2
t = np.arange(n_samples) / sfreq
for ch in ['O1', 'O2', 'Oz']:
    idx = ch_names.index(ch)
    data[idx] += 2e-6 * np.sin(2 * np.pi * 10 * t)  # 10 Hz

# Criar objeto Raw
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
raw = mne.io.RawArray(data, info)

# Set montage (posições dos eletrodos)
try:
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage)
    print("✅ Montage setada")
except Exception as e:
    print(f"⚠️ Erro ao setar montage: {e}")

print(f"\nRaw criado: {raw.n_times} samples, {raw.info['sfreq']} Hz, {len(raw.ch_names)} canais, {raw.times[-1]:.1f} s")

In [ ]:
# 2. Visualizar sinal bruto
print("\n=== Visualizando sinal bruto ===")

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
for i, ch in enumerate(['Fp1', 'C3', 'Pz', 'O1']):
    ch_idx = ch_names.index(ch)
    axes[i].plot(raw.times, raw._data[ch_idx] * 1e6, linewidth=0.5)
    axes[i].set_ylabel(f'{ch} (µV)')
    axes[i].grid(True, alpha=0.3)
axes[-1].set_xlabel('Tempo (s)')
fig.suptitle('Sinal bruto de EEG (4 canais representativos)', fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig('02_raw_signal.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 02_raw_signal.png")

In [ ]:
# 3. Criar eventos sintéticos (3 condições: palavra, pseudopalavra, falsa-fonte)
print("\n=== Criando eventos sintéticos ===")

n_events = 60  # 20 por condição
event_times = np.linspace(5, duration - 2, n_events)  # espaçados uniformemente
event_codes = np.tile([1, 2, 3], 20)  # 1=palavra, 2=pseudopalavra, 3=falsa-fonte
np.random.shuffle(event_codes)

events = np.zeros((n_events, 3), dtype=int)
events[:, 0] = (event_times * sfreq).astype(int)  # samples
events[:, 1] = 0  # sem offset
events[:, 2] = event_codes

print(f"Eventos criados: {n_events}")
print(f"  Condição 1 (palavras): {sum(event_codes == 1)}")
print(f"  Condição 2 (pseudopalavras): {sum(event_codes == 2)}")
print(f"  Condição 3 (falsas-fontes): {sum(event_codes == 3)}")

# Visualizar eventos no tempo
fig, ax = plt.subplots(figsize=(14, 3))
for ev in events:
    color = {1: 'blue', 2: 'orange', 3: 'gray'}[ev[2]]
    label = {1: 'palavra', 2: 'pseudopalavra', 3: 'falsa-fonte'}[ev[2]]
    ax.axvline(ev[0] / sfreq, color=color, alpha=0.6, linewidth=0.8)
ax.set_xlim(0, duration)
ax.set_xlabel('Tempo (s)')
ax.set_title('Eventos (palavras, pseudopalavras, falsas-fontes)')
plt.tight_layout()
plt.savefig('03_events.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 03_events.png")

In [ ]:
# 4. Aplicar filtros
print("\n=== Aplicando filtros ===")

raw_filt = raw.copy()
raw_filt.filter(l_freq=0.1, h_freq=30.0, method='fir', phase='zero')
raw_filt.notch_filter(freqs=[60.0])
print("✅ Filtros aplicados: 0.1-30 Hz + notch 60 Hz")

# Comparar antes/depois
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ch = 'O1'
ch_idx = ch_names.index(ch)
axes[0].plot(raw.times[:5000], raw._data[ch_idx, :5000] * 1e6, linewidth=0.5, color='gray')
axes[0].set_title(f'{ch} — ANTES do filtro')
axes[0].set_ylabel('µV')
axes[0].grid(True, alpha=0.3)

axes[1].plot(raw_filt.times[:5000], raw_filt._data[ch_idx, :5000] * 1e6, linewidth=0.5, color='blue')
axes[1].set_title(f'{ch} — DEPOIS do filtro (0.1-30 Hz)')
axes[1].set_xlabel('Tempo (s)')
axes[1].set_ylabel('µV')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_filter_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 04_filter_comparison.png")

In [ ]:
# 5. Criar Epochs
print("\n=== Criando Epochs ===")

event_id = {
    'palavra': 1,
    'pseudopalavra': 2,
    'falsa_fonte': 3
}

epochs = mne.Epochs(
    raw_filt,
    events,
    event_id=event_id,
    tmin=-0.2,
    tmax=1.0,
    baseline=(-0.2, 0.0),
    preload=True,
    reject=dict(eeg=100e-6)  # rejeitar trials com amplitude > 100 µV
)

print(f"\nEpochs criados:")
print(f"  Total de trials: {len(epochs)}")
print(f"  Rejeitados: {len(epochs.drop_log)}")
for cond in event_id.keys():
    print(f"  {cond}: {len(epochs[cond])}")

In [ ]:
# 6. Calcular ERPs e plotar
print("\n=== Calculando ERPs ===")

evokeds = {}
for cond in event_id.keys():
    evoked = epochs[cond].average()
    evokeds[cond] = evoked
    print(f"  {cond}: {len(epochs[cond])} trials")

# Plot ERPs em canais de interesse para cada componente
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (comp_name, config) in zip(axes, ERP_COMPONENTS.items()):
    chs = [ch for ch in config['channels'] if ch in ch_names]
    if not chs:
        continue
    ch_indices = [ch_names.index(ch) for ch in chs]
    times = list(evokeds.values())[0].times

    for cond, ev in evokeds.items():
        # Média dos canais
        mean_signal = ev.data[ch_indices].mean(axis=0) * 1e6
        ax.plot(times * 1000, mean_signal, label=cond, linewidth=2)

    # Sombrear janela do componente
    tmin_ms, tmax_ms = config['time_range'][0] * 1000, config['time_range'][1] * 1000
    ax.axvspan(tmin_ms, tmax_ms, alpha=0.2, color='gray',
               label=f"{comp_name}: {tmin_ms:.0f}-{tmax_ms:.0f} ms")
    ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
    ax.axvline(0, color='red', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_xlim(-200, 1000)
    ax.set_xlabel('Tempo (ms)')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f"{comp_name}\n{config['description']}")
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('05_erps_components.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 05_erps_components.png")

In [ ]:
# 7. Topografias
print("\n=== Plotando topografias ===")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cond in zip(axes, evokeds.keys()):
    times_ms = evokeds[cond].times * 1000
    # Pegar tempo no meio do N170 (170 ms)
    t_peak_ms = 170
    if t_peak_ms < times_ms[-1]:
        mne.viz.plot_topomap(
            evokeds[cond].data[:, (times_ms >= t_peak_ms - 5) & (times_ms <= t_peak_ms + 5)].mean(axis=1),
            evokeds[cond].info,
            axes=ax,
            show=False,
            time_format=''
        )
        ax.set_title(f'{cond}\n(N170, ~170 ms)', fontsize=10)

plt.suptitle('Topografia: N170 por condição', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('06_topography_n170.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 06_topography_n170.png")

In [ ]:
# 8. Espectro de potência (análise frequencial)
print("\n=== Análise espectral ===")

from scipy import signal

ch = 'O1'
ch_idx = ch_names.index(ch)
freqs, psd = signal.welch(raw._data[ch_idx], fs=sfreq, nperseg=4*sfreq)

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs, psd, color='blue', linewidth=1)
ax.set_xlim(0, 50)
ax.set_xlabel('Frequência (Hz)')
ax.set_ylabel('PSD (V²/Hz)')
ax.set_title(f'Densidade espectral de potência — {ch}')
ax.grid(True, alpha=0.3)
ax.axvline(10, color='red', linestyle='--', alpha=0.5, label='Alfa (8-12 Hz)')
ax.axvspan(8, 12, alpha=0.2, color='red')
ax.legend()
plt.tight_layout()
plt.savefig('07_psd.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 07_psd.png")

In [ ]:
# 9. Resumo: extrair métricas dos componentes
print("\n=== Métricas dos componentes ===\n")

metricas = []
for cond, ev in evokeds.items():
    for comp_name, config in ERP_COMPONENTS.items():
        chs = [ch for ch in config['channels'] if ch in ch_names]
        if not chs:
            continue
        ch_indices = [ch_names.index(ch) for ch in chs]
        tmin, tmax = config['time_range']
        time_mask = (ev.times >= tmin) & (ev.times <= tmax)
        mean_signal = ev.data[ch_indices].mean(axis=0)
        amp_media = mean_signal[time_mask].mean() * 1e6
        if comp_name.startswith('N'):
            amp_pico = mean_signal[time_mask].min() * 1e6
        else:
            amp_pico = mean_signal[time_mask].max() * 1e6
        metricas.append({
            'Condição': cond,
            'Componente': comp_name,
            'Canais': ', '.join(chs),
            'Amp_média (µV)': round(amp_media, 2),
            'Amp_pico (µV)': round(amp_pico, 2),
        })

df_metricas = pd.DataFrame(metricas)
print(df_metricas.to_string(index=False))

df_metricas.to_csv('08_metricas_componentes.csv', index=False)
print("\n✅ Salvo: 08_metricas_componentes.csv")

# Conclusões e próximos passos

## O que aprendemos

1. **MNE-Python** facilita o trabalho com EEG: criar dados sintéticos, filtrar, epochar, calcular ERPs
2. **Componentes principais** (N170, N400, P300, P600) podem ser visualizados em poucos passos
3. **Topografias** mostram a distribuição espacial da atividade cerebral
4. **Espectro de potência** revela picos de frequência (alfa, beta, etc.)

## Próximos passos

- [ ] Aplicar **ICA** para remover artefatos (olhos, músculo)
- [ ] Fazer **análise estatística** (cluster-based permutation test)
- [ ] Comparar **grupos** (experimental vs. controle) usando `mne.stats.spatio_temporal_cluster_*`
- [ ] Carregar **dados reais** do P03
- [ ] Integrar com **R** para SEM/LGCM

## Recursos

- [MNE Tutorials](https://mne.tools/mne-1.5/auto_tutorials/index.html)
- [Mike Cohen's EEG/ERP textbook](https://doi.org/10.1093/med/9780190224140.001.0001)
- [Luck (2014). An Introduction to the Event-Related Potential Technique](https://mitpress.mit.edu/9780262525855)